## Make table of pixel intensities for measured poweres 

### make imports

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
##### import project related moduls ####
current_file = Path.cwd() # cwd = path/*.ipynb - does not work in .py files.
print(f"current_file = {current_file}")
project_root = current_file.parent.parent
print(f"project_root = {project_root}")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import utils.file_utils as fu   
from utils.terminal_styler import TerminalColours as tc
from spf2_converter import BGCorrector

%matplotlib QtAgg

%load_ext autoreload
%autoreload 2

In [ ]:
%reload_ext autoreload

### Load and transform data

In [ ]:
corr = BGCorrector()

In [ ]:
corr.select_folder()
# folder_path=Path(r"C:\Andrei\DATA\VINETA_75\2026_09_11_spec\p4p5Pa_fast_scan\txt")
files = fu.list_files_in_folder(
    corr.data_dir
    )

data_A = {}
data_B = {}
wavelengths_map = None
i = 0
for file in files:
    # print(file.name)
    name_parts = file.name.split("_")
    power = name_parts[1].replace("p", ".").replace(".txt", "").replace("P", "").replace("kW", "")
    power = float(power)
    mindex = int(name_parts[0]) # measurement_index = mindex
    # print(name_parts)

    df = corr.read_file(file)
    ## set wavlength map for the first file only
    if wavelengths_map is None:
        wavelengths_map = pd.DataFrame({
            "pixel": df.index,
            "wavelength1": df["wavelength1"], 
            "wavelength2": df["wavelength2"]
            })


    row_A = df["intensity1"].copy()
   # print(row_A)
    row_A["power"] = power
    row_A["mindex"] = mindex
   # print(row_A)
    data_A[i] = row_A
   # print("+"*20)

    row_B = df["intensity2"].copy()
    row_B["power"] = power
    row_B["mindex"] = mindex
    data_B[i] = row_B
    i += 1

print(data_A.keys())
print(data_B.keys())

In [ ]:
def make_df(data:dict):
    df = pd.DataFrame.from_dict(data, orient="index").sort_index()
    first_cols = ["power", "mindex"]
    remaining_cols = df.columns.drop(first_cols).tolist()
    new_column_order = first_cols + remaining_cols
    return df[new_column_order].copy()


spectr_A = make_df(data_A)
spectr_B = make_df(data_B)

# print(wavelengths_map.head(2))
# print(spectr_A.head(2))

print(corr.data_dir)
output_folder = corr.data_dir.parent / "tables"
print(output_folder)
output_folder.mkdir(exist_ok=True)

table_A_path = output_folder / "spectr_A.csv"
table_B_path = output_folder / "spectr_B.csv"
table_map_path = output_folder / "wavelengths_map.csv"

spectr_A.to_csv(table_A_path, sep="\t", index=True)
spectr_B.to_csv(table_B_path, sep="\t", index=True)
wavelengths_map.to_csv(table_map_path, sep="\t", index=True)
